Agent结构化输出和前面4种差不多

1. 使用create_agent 的 response_format
2. 手动绑定OutputParser

使用response_format的方式，有三种策略：
1. ToolStrategy：把你的输出 schema封装成一工具，让模型调用
2. ProviderStrategy：利用模型提供商的原生的结构化输出API
3. AutoStrategy：create_agent的默认策略，它会检测当前模型支持哪种类型，自动的选择

模型生成结构化数据后，会将数据存入 structured_response 中


tool_message_content:自定义信息







In [8]:


from langchain.agents import create_agent
from langchain.agents.structured_output import AutoStrategy, ToolStrategy

from langchain.chat_models import init_chat_model
from pydantic import BaseModel, Field
from rich import print as rprint
import os
from dotenv import load_dotenv

load_dotenv(verbose=True)

api_key = os.getenv("COMPANY_API_KEY")
base_url = os.getenv("COMPANY_BASE_URL")

llm = init_chat_model(
    model="glm-5.3-flash",
    model_provider="openai",
    api_key=api_key,
    base_url=base_url
)




class Book(BaseModel):
    name:str = Field(description="图书名称")
    price:float =Field(description="图书价格")

graph = create_agent(
    model=llm,
    system_prompt= "你是个通用智能体",
    # 给智能体起个名字
    name="通用智能体",
    # 这是默认使用的AutoStrategy
    # response_format=Book
    # 自定义 tool_message_content消息
    response_format=ToolStrategy(Book,tool_message_content="已成功获取数据信息",handle_errors="请输入检查天气api是否正确")

)


response = graph.stream({
    "messages": [{"role":"user","content":"你好"}]
},stream_mode="values")




# response = graph.invoke(
#     {
#         "messages": [{"role": "user", "content": "把草房子这本书的信息给我"}]
#     }
# )

rprint(response)
# 返回的是一个messages列表，其中会包含 各种角色的消息类型，取最后一条模型的消息
print(response['messages'][-1].content)



<generator object Pregel.stream at 0x000001616203EDC0>

TypeError: 'generator' object is not subscriptable

错误处理：
AI应用开发中错误处理一般包含：
1. 结构化输出错误处理
2. 工具执行错误处理
3. 模型调用错误处理
4. 自定义异常错误处理

agent流式输出及模式："values", "updates", "checkpoints", "tasks", "debug", "messages", "custom"
1. updates（默认）：在每个Agent步骤结束后，流式返回该步骤产生的状态更新
2. messages:流式返回模型生成的Token元组，这是实现打字机效果
3. values：在每个步骤后，流式返回当前完整的状态快照
4. custom：允许你从工具或节点内部通过 write发出自定义信号，例如报告 已处理 10/100条记录




In [ ]:


from langchain.agents import create_agent
from langchain.agents.structured_output import AutoStrategy, ToolStrategy

from langchain.chat_models import init_chat_model
from pydantic import BaseModel, Field
from rich import print as rprint
import os
from dotenv import load_dotenv

load_dotenv(verbose=True)

api_key = os.getenv("COMPANY_API_KEY")
base_url = os.getenv("COMPANY_BASE_URL")

llm = init_chat_model(
    model="glm-5.3-flash",
    model_provider="openai",
    api_key=api_key,
    base_url=base_url
)




class Book(BaseModel):
    name:str = Field(description="图书名称")
    price:float =Field(description="图书价格")

graph = create_agent(
    model=llm,
    system_prompt= "你是个通用智能体",
    # 给智能体起个名字
    name="通用智能体",
    # 这是默认使用的AutoStrategy
    # response_format=Book
    # 自定义 tool_message_content消息
    response_format=ToolStrategy(Book,tool_message_content="已成功获取数据信息",handle_errors="请输入检查天气api是否正确")

)




# for chunk in graph.stream(
#     {"messages": [{"role": "user", "content": "你好"}]},
#     # stream_mode="values"
#     stream_mode="messages"
# ):
#     # 每个 chunk 是当前步骤后的完整状态快照
#     last_message = chunk["messages"][-1]
#     print(f"[{last_message.type}] {last_message.content}")


for message_chunk, metadata in graph.stream(
{"messages": [{"role": "user", "content": "你好"}]},
stream_mode="messages"
):
    # message_chunk 是 AIMessageChunk，逐 token 流式返回
    # metadata 包含该 token 来自哪个节点、哪次模型调用
    if message_chunk.content:
        print(message_chunk.content, end="", flush=True)

print()  # 换行







Failed to send compressed multipart ingest: langsmith.utils.LangSmithAuthError: Authentication failed for https://api.smith.langchain.com/runs/multipart. HTTPError('401 Client Error: Unauthorized for url: https://api.smith.langchain.com/runs/multipart', '{"error":"Unauthorized"}\n')trace=01a0e707-07f9-7321-b941-2ffc3b21bbec,id=01a0e707-07f9-7321-b941-2ffc3b21bbec; trace=01a0e707-07f9-7321-b941-2ffc3b21bbec,id=01a0e707-07fd-7c80-8219-a993da0c4760; trace=01a0e707-07f9-7321-b941-2ffc3b21bbec,id=01a0e707-07fe-7533-8593-df24f488406b


你好！😊 很高兴见到你！

我可以帮你处理各种任务，比如：

- **登记图书信息**：告诉我图书的名称和价格，我可以帮你录入
- **回答问题**：知识咨询、信息查询等
- **文本处理**：写作、翻译、总结等

请问有什么可以帮到你的吗？

Failed to send compressed multipart ingest: langsmith.utils.LangSmithAuthError: Authentication failed for https://api.smith.langchain.com/runs/multipart. HTTPError('401 Client Error: Unauthorized for url: https://api.smith.langchain.com/runs/multipart', '{"error":"Unauthorized"}\n')trace=01a0e707-07f9-7321-b941-2ffc3b21bbec,id=01a0e707-07fe-7533-8593-df24f488406b; trace=01a0e707-07f9-7321-b941-2ffc3b21bbec,id=01a0e707-07fd-7c80-8219-a993da0c4760; trace=01a0e707-07f9-7321-b941-2ffc3b21bbec,id=01a0e707-ba7c-7f40-9d20-1c4da6ecdc4b; trace=01a0e707-07f9-7321-b941-2ffc3b21bbec,id=01a0e707-ba7d-7601-9902-dc3ff81ad591


你好！👋 很高兴见到你！

我可以帮你做很多事情，比如：

- **📚 登记图书**：告诉我书名和价格，我可以帮你录入图书信息
- **💬 解答问题**：各类知识咨询和信息查询
- **✍️ 文字处理**：写作、翻译、总结润色等

请问今天有什么可以帮到你的呢？

Failed to send compressed multipart ingest: langsmith.utils.LangSmithAuthError: Authentication failed for https://api.smith.langchain.com/runs/multipart. HTTPError('401 Client Error: Unauthorized for url: https://api.smith.langchain.com/runs/multipart', '{"error":"Unauthorized"}\n')trace=01a0e707-07f9-7321-b941-2ffc3b21bbec,id=01a0e707-ba7d-7601-9902-dc3ff81ad591; trace=01a0e707-07f9-7321-b941-2ffc3b21bbec,id=01a0e707-ba7c-7f40-9d20-1c4da6ecdc4b; trace=01a0e707-07f9-7321-b941-2ffc3b21bbec,id=01a0e708-19d1-7480-ac00-4d5fdf918f6e; trace=01a0e707-07f9-7321-b941-2ffc3b21bbec,id=01a0e708-19d2-7cb1-9faf-3b7878e38ed3


...

你好！😊 很高兴见到你。

有什么可以帮你的吗？直接告诉我你的需求就好：

- **📚 登记图书**：提供书名和价格，我来帮你录入
- **💬 咨询问题**：各类知识问答
- **✍️ 文字任务**：写作、翻译、总结等

期待为你服务～

Failed to send compressed multipart ingest: langsmith.utils.LangSmithAuthError: Authentication failed for https://api.smith.langchain.com/runs/multipart. HTTPError('401 Client Error: Unauthorized for url: https://api.smith.langchain.com/runs/multipart', '{"error":"Unauthorized"}\n')trace=01a0e707-07f9-7321-b941-2ffc3b21bbec,id=01a0e772-167b-7c41-a377-19ed0a2f6cc3; trace=01a0e707-07f9-7321-b941-2ffc3b21bbec,id=01a0e772-1674-74e2-b74c-6ffbfe5e51cf; trace=01a0e707-07f9-7321-b941-2ffc3b21bbec,id=01a0e772-48a8-7ea2-99d2-82271d11b9db; trace=01a0e707-07f9-7321-b941-2ffc3b21bbec,id=01a0e772-48b3-7e10-bff6-45c9e4efe964


你好！👋 我是你的智能助手，很高兴见到你。

我可以帮你：

- **📚 登记图书**：告诉我书名和价格，我来帮你录入
- **💬 解答问题**：各类知识咨询
- **✍️ 文字任务**：写作、翻译、总结等

有什么需求，直接告诉我即可～

Failed to send compressed multipart ingest: langsmith.utils.LangSmithAuthError: Authentication failed for https://api.smith.langchain.com/runs/multipart. HTTPError('401 Client Error: Unauthorized for url: https://api.smith.langchain.com/runs/multipart', '{"error":"Unauthorized"}\n')trace=01a0e707-07f9-7321-b941-2ffc3b21bbec,id=01a0e772-48b3-7e10-bff6-45c9e4efe964; trace=01a0e707-07f9-7321-b941-2ffc3b21bbec,id=01a0e772-48a8-7ea2-99d2-82271d11b9db; trace=01a0e707-07f9-7321-b941-2ffc3b21bbec,id=01a0e772-6c88-70b0-884b-96fda26b43c3; trace=01a0e707-07f9-7321-b941-2ffc3b21bbec,id=01a0e772-6c8f-7432-a92a-7a9f8a3bd970


你好！😊 我是你的智能助手，很高兴见到你。

有什么可以帮你的吗？比如：

- **📚 登记图书**：告诉我书名和价格，我来帮你录入
- **💬 咨询问题**：各类知识问答
- **✍️ 文字任务**：写作、翻译、总结等

直接告诉我你的需求就好～